# 🏛️ Clase 4 · Caso real: Triage inteligente de PQRS con Transformers

**Módulo:** Científico de Datos e Inteligencia Artificial Aplicada · **Unidad 2:** NLP y el Auge de la IA Generativa (LLMs)

---
## 📌 El problema (real)

**ServiRed S.A. E.S.P.** (empresa ficticia de energía e internet, problema 100 % real) recibe **~3.000 PQRS al mes**
(Peticiones, Quejas, Reclamos, Sugerencias, Denuncias) por web, correo, app y teléfono.

- 👥 **4 analistas** leen cada mensaje, deciden el **tipo**, el **área** responsable y la **urgencia**.
- ⏱️ Cada lectura + clasificación toma ~6 min → **300 horas/mes** solo en clasificar.
- ⚖️ **Ley 142 de 1994 (art. 158):** la empresa tiene **15 días hábiles** para responder. Si no lo hace,
  opera el **silencio administrativo positivo**: se entiende que **el usuario tiene la razón**.
- 🚨 Los casos graves (una persona con oxígeno sin luz, un poste a punto de caer) se pierden entre sugerencias y felicitaciones.
- 🔁 Cuando hay un daño masivo, llegan 200 mensajes del mismo problema y cada uno se trata por separado.

## 🎯 Lo que vamos a construir
```
PQRS ─► [Transformer] entiende el texto
          ├─ ¿Qué TIPO es?            (zero-shot NLI)
          ├─ ¿Qué ÁREA lo atiende?    (zero-shot NLI)
          ├─ ¿Qué tan URGENTE es?     (sentimiento + señales de riesgo)
          ├─ ¿Quién, dónde, cuánto?   (NER + regex) ─► anonimizar datos personales
          ├─ ¿Es un caso MASIVO?      (embeddings + clustering)
          ├─ ¿POR QUÉ decidió eso?    (explicabilidad por oclusión)
          └─ Borrador de respuesta    (LLM con los datos extraídos)
       ─► Bandeja priorizada con fecha límite legal ─► el analista revisa y aprueba (humano en el loop)
```

### ⏱️ Agenda
| Bloque | Tema |
|---|---|
| 1 | ¿Por qué no basta con reglas (`if "queja" in texto`)? |
| 2 | **¿Cómo "lee" un Transformer?** tokens, embeddings contextuales, atención, decisión |
| 3 | La solución paso a paso (tipo, área, urgencia, entidades, masivos, explicabilidad, respuesta) |
| 4 | Bandeja priorizada, impacto y riesgos |

> ## 🧑‍🏫 Guía para el docente (no aparece en el notebook de estudiantes)
> - **Mensaje del día:** la IA no "reemplaza al analista": **lee, ordena y prepara**; el humano decide. El valor está en el **sistema**, no en un modelo.
> - **Antes de clase:** ejecuta las secciones 0 y 2 con internet. Modelos: BETO (~440 MB), zero-shot XNLI (~440 MB), RoBERTuito (~435 MB), NER (~420 MB), MiniLM (~470 MB) y Qwen3-0.6B (~1,5 GB, ya está de la Clase 3).
> - **Abre con una pregunta:** "¿Quién ha puesto una PQRS? ¿Cuánto tardaron en responder?". Conecta con su experiencia.
> - **La sección 2 es el corazón conceptual:** tómate el tiempo con el ejemplo de la palabra "corte" (2.2) y el mapa de atención (2.3).
> - **Tiempos CPU:** zero-shot sobre 38 PQRS ≈ 30-60 s por tarea; oclusión ≈ 10 s por mensaje; Qwen3 ≈ 5-15 s por respuesta.

---
## 0. Preparación

In [ ]:
import importlib.util, subprocess, sys
paquetes = {"torch": "torch", "transformers": "transformers>=4.51", "sentence_transformers": "sentence-transformers",
            "sklearn": "scikit-learn", "pandas": "pandas", "matplotlib": "matplotlib", "accelerate": "accelerate"}
faltan = [p for m, p in paquetes.items() if importlib.util.find_spec(m) is None]
if faltan:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
print("✓ Entorno listo" + (f" (instalado: {faltan})" if faltan else ""))

In [ ]:
import html, re, time, warnings
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from IPython.display import HTML, display
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, pipeline

warnings.filterwarnings("ignore"); transformers.logging.set_verbosity_error()
pd.set_option("display.max_colwidth", 140)
torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODELO_BETO = "dccuchile/bert-base-spanish-wwm-cased"            # para "abrir" el Transformer
MODELO_ZS   = "Recognai/bert-base-spanish-wwm-cased-xnli"        # zero-shot (NLI)
MODELO_SENT = "pysentimiento/robertuito-sentiment-analysis"      # sentimiento
MODELO_NER  = "mrm8488/bert-spanish-cased-finetuned-ner"         # entidades
MODELO_EMB  = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODELO_LLM  = "Qwen/Qwen3-0.6B"                                  # borradores de respuesta
GENERAR_RESPUESTAS = True                                        # False si no hay tiempo/memoria

pqrs = pd.read_csv(Path("..") / "datasets" / "pqrs_servired.csv", parse_dates=["fecha"])
print(device, "|", len(pqrs), "PQRS"); pqrs.sample(6, random_state=1)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3))
for a, col in zip(ax, ["tipo", "area", "urgencia"]):
    pqrs[col].value_counts().plot.barh(ax=a, color="#7B2CBF", title=f"{col} (etiqueta humana)")
plt.tight_layout(); plt.show()

---
## 1. ¿Por qué no basta con reglas?
La primera idea de cualquier equipo: palabras clave. Probemos.

In [ ]:
# ✍️ LIVE CODING — 1.1 Clasificador de tipo con reglas (palabras clave)
REGLAS = {"queja": ["queja", "pésimo", "mal servicio", "nunca"], "reclamo": ["cobro", "factura", "cobraron"],
          "peticion": ["solicito", "quisiera", "quiero saber"], "sugerencia": ["sería bueno", "deberían"],
          "denuncia": ["denuncio", "ilegal"], "felicitacion": ["felicitar", "gracias", "amable"]}

def tipo_por_reglas(texto):
    t = texto.lower()
    for tipo, palabras in REGLAS.items():
        if any(p in t for p in palabras):
            return tipo
    return "peticion"

pqrs["tipo_reglas"] = pqrs.texto.apply(tipo_por_reglas)
print(f"Exactitud con reglas: {accuracy_score(pqrs.tipo, pqrs.tipo_reglas):.0%}")
pqrs.loc[pqrs.tipo != pqrs.tipo_reglas, ["texto", "tipo", "tipo_reglas"]].head(8)

**¿Por qué fallan?**
| Caso | Texto | Problema |
|---|---|---|
| Negación | *"**No es una queja**, solo quiero saber…"* | La regla ve "queja" y no entiende el "no" |
| Sinónimos | *"nadie nos da respuesta"*, *"se nos dañó la comida"* | Nadie escribió "queja" |
| Contexto | *"Me **cobraron** la reconexión dos veces"* vs *"un funcionario está **cobrando** en efectivo"* | Misma palabra: reclamo vs **denuncia** |
| Mantenimiento | Cada caso nuevo = una regla más | Cientos de reglas que se contradicen |

👉 Necesitamos un modelo que entienda **significado y contexto**, no letras. Eso es un **Transformer**.

---
## 2. ¿Cómo "lee" un Transformer? (abrimos la caja negra)
```
"Me cobraron la reconexión"
   │ 1. Tokenizer:   [CLS] Me cobr ##aron la recon ##exión [SEP]       ← texto → piezas → números
   │ 2. Embeddings:  cada token → vector de 768 números                 ← significado "de diccionario"
   │ 3. 12 capas de SELF-ATTENTION: cada token mira a los demás          ← el CONTEXTO cambia los vectores
   │ 4. Vector [CLS] final = resumen de toda la frase
   ▼ 5. Cabeza de clasificación + softmax → "reclamo: 0.87"
```

In [ ]:
# ✍️ LIVE CODING — 2.1 Paso 1: el tokenizer (texto → piezas → números)
tok = AutoTokenizer.from_pretrained(MODELO_BETO)
beto = AutoModel.from_pretrained(MODELO_BETO, output_attentions=True, output_hidden_states=True,
                                 attn_implementation="eager").to(device).eval()
frase = "Me cobraron la reconexión aunque pagué la factura a tiempo"
enc = tok(frase, return_tensors="pt")

def indice(toks, palabra):
    # posición del primer token (o sub-token) con el que empieza la palabra
    for i, t in enumerate(toks):
        if t not in ("[CLS]", "[SEP]") and len(t) >= 2 and palabra.lower().startswith(t.replace("##", "").lower()):
            return i
    raise ValueError(f"{palabra!r} no está en {toks}")

print("Piezas:", tok.convert_ids_to_tokens(enc["input_ids"][0]))
print("IDs:   ", enc["input_ids"][0].tolist())
print(f"\nBETO: {sum(p.numel() for p in beto.parameters())/1e6:.0f} M parámetros · "
      f"{beto.config.num_hidden_layers} capas · {beto.config.num_attention_heads} cabezas de atención · vectores de {beto.config.hidden_size}")

### 2.2 Paso 2-3: el **contexto** cambia el significado
En Word2Vec (Clase 1) cada palabra tenía **un solo vector**. En un Transformer, el vector de una palabra **depende de la frase**.
Prueba con una palabra clave en PQRS: **"corte"**.

In [ ]:
# ✍️ LIVE CODING — 2.2 La misma palabra "corte" en 4 contextos: vector estático vs contextual
frases = ["Me hicieron un corte del servicio sin aviso",
          "Hubo un corte de energía en todo el barrio",
          "La fecha de corte de la factura es el 15",
          "Me hice un corte de pelo nuevo"]

def vectores_de(palabra, frase):
    e = tok(frase, return_tensors="pt").to(device)
    idx = indice(tok.convert_ids_to_tokens(e["input_ids"][0]), palabra)
    with torch.no_grad():
        out = beto(**e)
    return [h[0, idx].cpu() for h in out.hidden_states]          # vector en cada capa (0 = entrada … 12 = salida)

capas = [vectores_de("corte", f) for f in frases]
estatico = torch.stack([c[0] for c in capas]); contextual = torch.stack([c[-1] for c in capas])
sim = lambda X: F.cosine_similarity(X.unsqueeze(1), X.unsqueeze(0), dim=-1).numpy()

etiquetas = ["corte del servicio", "corte de energía", "fecha de corte", "corte de pelo"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for a, M_, t in zip(ax, [sim(estatico), sim(contextual)], ["Capa 0 (casi como Word2Vec)", "Capa 12 (después de la atención)"]):
    im = a.imshow(M_, cmap="Purples", vmin=0.3, vmax=1); a.set_title(t)
    a.set_xticks(range(4)); a.set_xticklabels(etiquetas, rotation=30, ha="right"); a.set_yticks(range(4)); a.set_yticklabels(etiquetas)
    for i in range(4):
        for j in range(4): a.text(j, i, f"{M_[i, j]:.2f}", ha="center", va="center", fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
# ✍️ LIVE CODING — 2.3 ¿En qué capa aparece el significado? similitud con "corte del servicio" capa por capa
plt.figure(figsize=(8, 3.5))
for k, nombre in [(1, "corte de energía"), (2, "fecha de corte"), (3, "corte de pelo")]:
    plt.plot([F.cosine_similarity(capas[0][l], capas[k][l], dim=0).item() for l in range(13)], "o-", label=nombre)
plt.xlabel("capa del Transformer"); plt.ylabel("similitud con 'corte del servicio'")
plt.title("El contexto se construye capa a capa"); plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

> 🧑‍🏫 **Qué deben notar:** en la capa 0 las cuatro "corte" son casi idénticas (solo cambia la posición).
> En la capa 12, "corte del servicio" y "corte de energía" quedan **cerca** y "corte de pelo" se **aleja**.
> Eso es lo que permite que el modelo diferencie un reclamo técnico de un tema irrelevante **sin que nadie escriba una regla**.

### 2.4 Self-attention: ¿a qué palabras "mira" cada palabra?
En cada capa, cada token calcula **cuánto atender** a los demás:
$\text{Atención}(Q,K,V)=\text{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)V$.
Así *"pagué"* puede conectar con *"cobraron"* y el modelo "entiende" que hay una **contradicción** (señal de reclamo).

In [ ]:
# ✍️ LIVE CODING — 2.4 Mapa de atención (promedio de las 12 cabezas) en una capa
def mapa_atencion(frase, capa=8):
    e = tok(frase, return_tensors="pt").to(device)
    with torch.no_grad():
        att = beto(**e).attentions[capa][0].mean(0).cpu().numpy()     # [tokens, tokens]
    toks = tok.convert_ids_to_tokens(e["input_ids"][0])
    plt.figure(figsize=(7.5, 6.5)); plt.imshow(att, cmap="Purples")
    plt.xticks(range(len(toks)), toks, rotation=60); plt.yticks(range(len(toks)), toks)
    plt.title(f"Atención · capa {capa} (fila = quién mira, columna = a quién)"); plt.colorbar(fraction=.04)
    plt.tight_layout(); plt.show()
    return toks, att

toks, att = mapa_atencion(frase)

In [ ]:
# ✍️ LIVE CODING — 2.5 ¿A qué mira una palabra específica?
def a_que_mira(palabra, toks=toks, att=att, top=5):
    i = indice(toks, palabra)
    orden = [j for j in np.argsort(-att[i]) if toks[j] not in ("[CLS]", "[SEP]") and j != i][:top]
    pd.Series(att[i][orden], index=[toks[j] for j in orden]).plot.barh(color="#9D4EDD", figsize=(5, 2.5),
                                                                       title=f"'{palabra}' presta atención a…")
    plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

a_que_mira("pagué")

### 2.5 Paso 5: de vectores a una decisión — ¿cómo clasifica **sin entrenarlo** (zero-shot)?
Usamos un Transformer entrenado en **NLI** (*Natural Language Inference*): dada una **premisa** y una **hipótesis**, decide si la premisa la **implica**.
```
premisa:   "Me cobraron la reconexión aunque pagué la factura a tiempo"
hipótesis: "Este mensaje es un reclamo por un cobro o una factura."   → implica: 0.91 ✅
hipótesis: "Este mensaje es una felicitación o agradecimiento."      → implica: 0.02 ❌
```
Cambiamos las etiquetas **escribiendo frases**, sin reentrenar nada. Por eso sirve el primer día, sin datos etiquetados.

In [ ]:
# ✍️ LIVE CODING — 2.6 Ver el NLI "por dentro": logits de contradicción / neutral / implicación
tok_zs = AutoTokenizer.from_pretrained(MODELO_ZS)
nli = transformers.AutoModelForSequenceClassification.from_pretrained(MODELO_ZS).to(device).eval()
print(nli.config.id2label)
for h in ["Este mensaje es un reclamo por un cobro o una factura.", "Este mensaje es una felicitación o agradecimiento."]:
    e = tok_zs(frase, h, return_tensors="pt").to(device)                # premisa [SEP] hipótesis
    with torch.no_grad():
        p = F.softmax(nli(**e).logits, -1)[0].cpu().numpy()
    print(f"{h:<58} → " + " · ".join(f"{nli.config.id2label[i]}={v:.2f}" for i, v in enumerate(p)))

---
## 3. La solución paso a paso
### 3.1 ¿Qué TIPO de PQRS es? (zero-shot)
👉 **Las etiquetas son frases descriptivas**: aquí está el "prompt engineering" del zero-shot.

In [ ]:
# ✍️ LIVE CODING — 3.1 Clasificar el tipo de todas las PQRS sin entrenar
zero_shot = pipeline("zero-shot-classification", model=MODELO_ZS, device=device)
TIPOS = {"peticion": "una solicitud de información o de un trámite",
         "queja": "una queja por una falla o por mala atención",
         "reclamo": "un reclamo para que corrijan un cobro o reparen un daño",
         "sugerencia": "una sugerencia para mejorar",
         "denuncia": "una denuncia de fraude o de un acto ilegal",
         "felicitacion": "una felicitación o un agradecimiento"}
PLANTILLA = "Este mensaje es {}."

def clasificar(textos, etiquetas):
    inv = {v: k for k, v in etiquetas.items()}
    res = zero_shot(list(textos), candidate_labels=list(etiquetas.values()), hypothesis_template=PLANTILLA)
    return [inv[r["labels"][0]] for r in res], [round(r["scores"][0], 3) for r in res]

t = time.time()
pqrs["tipo_ia"], pqrs["conf_tipo"] = clasificar(pqrs.texto, TIPOS)
print(f"⏱️ {time.time()-t:.0f}s · Exactitud IA: {accuracy_score(pqrs.tipo, pqrs.tipo_ia):.0%} "
      f"vs reglas: {accuracy_score(pqrs.tipo, pqrs.tipo_reglas):.0%}")
ConfusionMatrixDisplay.from_predictions(pqrs.tipo, pqrs.tipo_ia, cmap="Purples", xticks_rotation=45)
plt.title("Tipo de PQRS · zero-shot"); plt.tight_layout(); plt.show()

In [ ]:
# ✍️ LIVE CODING — 3.2 ¿Qué ÁREA lo atiende?
AREAS = {"facturacion": "sobre facturas, cobros, pagos o deudas",
         "tecnica": "sobre fallas técnicas, cortes de energía, internet o instalaciones",
         "atencion_cliente": "sobre la atención al cliente, oficinas o canales de contacto",
         "juridica": "sobre temas legales, fraudes, derechos o datos personales"}
pqrs["area_ia"], pqrs["conf_area"] = clasificar(pqrs.texto, AREAS)
print(f"Exactitud área: {accuracy_score(pqrs.area, pqrs.area_ia):.0%}")
pqrs.loc[(pqrs.tipo != pqrs.tipo_ia) | (pqrs.area != pqrs.area_ia), ["texto", "tipo", "tipo_ia", "area", "area_ia"]]

> 🧑‍🏫 **Discusión sobre los errores:** muchos son **ambiguos incluso para humanos** (¿un poste caído es queja o reclamo?).
> En la vida real se mide el **acuerdo entre analistas** (suele estar en 80-90 %): esa es la meta, no el 100 %.
> **Mejora en vivo:** cambia una descripción en `TIPOS` y vuelve a correr. Cuando la empresa acumule ~50 ejemplos por clase,
> se hace **fine-tuning** de BETO (como en la clase regular) y la exactitud sube.

### 3.3 ¿Por qué decidió eso? **Explicabilidad por oclusión**
Un analista (o un juez) va a preguntar *"¿por qué la IA dijo que esto es una denuncia?"*.
Técnica simple y honesta: **quitamos una palabra a la vez** y medimos cuánto baja la probabilidad. Las palabras que más la bajan son las que **el modelo usó**.

In [ ]:
# ✍️ LIVE CODING — 3.3 Oclusión: importancia de cada palabra para la etiqueta elegida
def explicar(texto, etiquetas=TIPOS):
    desc = list(etiquetas.values())
    base = zero_shot(texto, candidate_labels=desc, hypothesis_template=PLANTILLA)
    objetivo, p0 = base["labels"][0], base["scores"][0]
    palabras = texto.split()
    variantes = [" ".join(palabras[:i] + palabras[i+1:]) for i in range(len(palabras))]
    res = zero_shot(variantes, candidate_labels=desc, hypothesis_template=PLANTILLA)
    imp = np.array([p0 - dict(zip(r["labels"], r["scores"]))[objetivo] for r in res])
    m = max(imp.max(), 1e-6)
    spans = "".join(f'<span style="background:rgba(123,44,191,{max(v, 0)/m:.2f});padding:2px 3px;border-radius:3px;'
                    f'color:{"#fff" if v/m > .5 else "inherit"}">{html.escape(w)}</span> ' for w, v in zip(palabras, imp))
    clave = [k for k, v in etiquetas.items() if v == objetivo][0]
    display(HTML(f'<div style="line-height:2.1;margin:6px 0"><b>{clave}</b> ({p0:.0%}) → {spans}</div>'))

for i in [5, 26, 37]:          # denuncia de cobro en efectivo · cobro doble · "No es una queja…"
    explicar(pqrs.texto[i])

### 3.4 ¿Qué tan URGENTE es? Modelo + conocimiento del negocio
La urgencia no es solo "qué tan enojado está". Combinamos:
- 🤖 **Sentimiento negativo** (RoBERTuito)
- 📋 **Señales de riesgo** que define el negocio (persona vulnerable, riesgo físico, riesgo legal…)
- 🔁 **Caso masivo** (lo calculamos en 3.6)

👉 Patrón profesional: **IA + reglas de negocio explicables**, no IA sola.

In [ ]:
# ✍️ LIVE CODING — 3.4 Sentimiento + señales de riesgo
sentimiento = pipeline("text-classification", model=MODELO_SENT, device=device, top_k=None)
pqrs["p_negativo"] = [next(s["score"] for s in r if s["label"] == "NEG") for r in sentimiento(list(pqrs.texto))]

SENALES = {r"ox[ií]geno|nebulizador|discapacidad|adulto mayor|menor|niñ[oa]s?|medicina": "persona vulnerable",
           r"cables? suelto|poste|quemado|incendi|chispa": "riesgo físico",
           r"tutela|demanda|superintendencia|derecho de petici[oó]n": "riesgo legal",
           r"datacr[eé]dito|reportar|amenaz": "riesgo crediticio",
           r"\b(dos|tres|\d+) d[ií]as|\d+ horas|desde el lunes": "afectación prolongada",
           r"efectivo|ilegal|vendi[oó] .*datos|sin mi autorizaci[oó]n": "posible fraude"}
pqrs["senales"] = pqrs.texto.apply(lambda t: [n for p, n in SENALES.items() if re.search(p, t, re.I)])
pqrs[["texto", "p_negativo", "senales"]].sort_values("p_negativo", ascending=False).head(8)

### 3.5 ¿Quién, dónde, cuánto? Entidades + **anonimización** (Habeas Data)
Antes de mandar un texto a un LLM o a un tablero, se **protegen los datos personales** (Ley 1581 de 2012).
NER encuentra **personas y lugares**; las expresiones regulares encuentran **cédulas, dinero y direcciones**.

In [ ]:
# ✍️ LIVE CODING — 3.5 NER + regex → entidades y texto anonimizado
ner = pipeline("token-classification", model=MODELO_NER, aggregation_strategy="simple", device=device)
REGEX = {"CEDULA": r"\b\d{1,3}(?:\.\d{3}){2,3}\b", "DINERO": r"\$\s?\d{1,3}(?:\.\d{3})*",
         "DIRECCION": r"\b(?:Calle|Carrera|Cra|Cl|Av)\.?\s*\d+[A-Za-z]?\s*(?:#|con|No\.?)\s*(?:Calle\s*)?\d+(?:-\d+)?"}

def extraer(texto):
    ents = [(e["entity_group"], e["word"], e["start"], e["end"]) for e in ner(texto) if e["score"] > .6]
    for tipo, patron in REGEX.items():
        ents += [(tipo, m.group(), m.start(), m.end()) for m in re.finditer(patron, texto)]
    anon, limite = texto, len(texto) + 1
    for tipo, _, s, e in sorted(ents, key=lambda x: -x[2]):
        if tipo in ("PER", "CEDULA", "DIRECCION") and e <= limite:     # evita reemplazos solapados
            anon, limite = anon[:s] + f"[{tipo}]" + anon[e:], s
    return {"personas": [w for t, w, *_ in ents if t == "PER"], "lugares": [w for t, w, *_ in ents if t == "LOC"],
            "dinero": [w for t, w, *_ in ents if t == "DINERO"], "texto_anonimo": anon}

ext = pd.DataFrame([extraer(t) for t in pqrs.texto])
pqrs = pd.concat([pqrs.drop(columns=[c for c in ext.columns if c in pqrs]), ext], axis=1)
pqrs.loc[pqrs.personas.str.len().gt(0) | pqrs.dinero.str.len().gt(0), ["texto_anonimo", "personas", "lugares", "dinero"]]

### 3.6 ¿Es un caso MASIVO? Embeddings + clustering
Si 50 usuarios reportan *"sin luz en Kennedy"* con palabras distintas, es **un solo problema**: se atiende **una vez** y se responde a todos.
Los **embeddings de oraciones** ponen cerca los textos con el mismo significado.

In [ ]:
# ✍️ LIVE CODING — 3.6 Agrupar PQRS por significado
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

embedder = SentenceTransformer(MODELO_EMB, device=str(device))
E = embedder.encode(list(pqrs.texto), normalize_embeddings=True, show_progress_bar=False)
grupos = AgglomerativeClustering(n_clusters=None, distance_threshold=0.45, metric="cosine", linkage="average").fit_predict(E)
pqrs["grupo"] = grupos
tam = pqrs.grupo.map(pqrs.grupo.value_counts())
pqrs["masivo"] = tam >= 3
for g, d in pqrs[pqrs.masivo].groupby("grupo"):
    print(f"🔁 Posible falla masiva (grupo {g}, {len(d)} PQRS):"); [print("   ·", t) for t in d.texto]

orden = np.argsort(grupos)
plt.figure(figsize=(6, 5)); plt.imshow(E[orden] @ E[orden].T, cmap="Purples", vmin=0, vmax=1)
plt.title("Similitud entre PQRS (ordenadas por grupo)"); plt.colorbar(fraction=.04); plt.tight_layout(); plt.show()

In [ ]:
# ✍️ LIVE CODING — 3.7 Puntaje de urgencia y evaluación contra los analistas
def urgencia(fila):
    puntaje = 0.35 * fila.p_negativo + 0.25 * min(len(fila.senales), 2) + 0.2 * fila.masivo
    puntaje += 0.3 * any(s in ("persona vulnerable", "riesgo físico") for s in fila.senales)
    return round(puntaje, 2)

pqrs["puntaje_urgencia"] = pqrs.apply(urgencia, axis=1)
pqrs["urgencia_ia"] = pd.cut(pqrs.puntaje_urgencia, [-1, 0.3, 0.6, 9], labels=["baja", "media", "alta"]).astype(str)
print(f"Exactitud urgencia: {accuracy_score(pqrs.urgencia, pqrs.urgencia_ia):.0%}")
altas = pqrs.urgencia == "alta"
print(f"Recall de urgencia ALTA (lo que más importa no perder): {(pqrs.urgencia_ia[altas] == 'alta').mean():.0%}")
print(classification_report(pqrs.urgencia, pqrs.urgencia_ia, zero_division=0))

> 🧑‍🏫 **Métrica correcta:** aquí importa más el **recall de la clase "alta"** que la exactitud global: perder un caso de oxígeno es mucho peor
> que marcar como urgente una sugerencia. Pregunta a la clase: *"¿Qué pesos cambiarían para no perder ningún caso alto?"*.

### 3.8 Borrador de respuesta con un LLM (con datos **anonimizados** y reglas)
El LLM **no decide**: redacta un borrador con lo que ya extrajimos. El analista lo revisa y lo firma.

In [ ]:
# ✍️ LIVE CODING — 3.8 Qwen3 redacta el borrador
if GENERAR_RESPUESTAS:
    tok_llm = AutoTokenizer.from_pretrained(MODELO_LLM)
    llm = AutoModelForCausalLM.from_pretrained(MODELO_LLM, dtype="auto" if device.type == "cuda" else torch.float32).to(device).eval()
    SISTEMA = ("Eres analista de PQRS de ServiRed S.A. E.S.P. Redacta un borrador de respuesta formal y empático en español, "
               "máximo 4 frases. No inventes cifras, fechas ni compromisos que no estén en los datos. "
               "Menciona el número de radicado y que la empresa responderá de fondo dentro del plazo legal.")

    def borrador(fila):
        datos = (f"Radicado: {fila.radicado}\nTipo: {fila.tipo_ia}\nÁrea: {fila.area_ia}\nUrgencia: {fila.urgencia_ia}\n"
                 f"Caso masivo: {'sí' if fila.masivo else 'no'}\nMensaje del usuario: {fila.texto_anonimo}")
        msgs = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": datos}]
        p = tok_llm.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        e = tok_llm(p, return_tensors="pt").to(device)
        with torch.no_grad():
            out = llm.generate(**e, max_new_tokens=140, do_sample=False, pad_token_id=tok_llm.eos_token_id)
        return tok_llm.decode(out[0, e["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    for i in [0, 1, 19]:
        f = pqrs.iloc[i]
        print(f"📨 {f.texto_anonimo}\n📝 {borrador(f)}\n")

---
## 4. Resultado: la bandeja priorizada
Cada PQRS con: tipo, área, urgencia, señales, si es masivo y **fecha límite legal** (15 días hábiles, sin festivos).

In [ ]:
# 4.1 Fecha límite legal y bandeja ordenada (viene lista)
FESTIVOS = ["2026-11-02", "2026-11-16", "2026-12-08", "2026-12-25", "2027-01-01", "2027-01-11"]
pqrs["fecha_limite"] = pd.to_datetime(np.busday_offset(pqrs.fecha.values.astype("datetime64[D]"), 15,
                                                       roll="forward", holidays=FESTIVOS))
HOY = pd.Timestamp("2026-11-24")
pqrs["dias_habiles_restantes"] = np.busday_count(np.datetime64(HOY.date()), pqrs.fecha_limite.values.astype("datetime64[D]"),
                                                 holidays=FESTIVOS)
prioridad = {"alta": 0, "media": 1, "baja": 2}
bandeja = pqrs.assign(_p=pqrs.urgencia_ia.map(prioridad)).sort_values(["_p", "dias_habiles_restantes", "puntaje_urgencia"],
                                                                       ascending=[True, True, False])
cols = ["radicado", "urgencia_ia", "tipo_ia", "area_ia", "masivo", "senales", "dias_habiles_restantes", "texto_anonimo"]
def color(v):
    return {"alta": "background:#F3D9FA;font-weight:bold", "media": "background:#EDE7F6"}.get(v, "")
bandeja[cols].head(15).style.map(color, subset=["urgencia_ia"])

In [ ]:
# 4.2 Tablero para el jefe de PQRS
fig, ax = plt.subplots(1, 3, figsize=(15, 3.5))
pd.crosstab(pqrs.area_ia, pqrs.urgencia_ia)[["alta", "media", "baja"]].plot.barh(stacked=True, ax=ax[0],
            color=["#7B2CBF", "#B197FC", "#E5DBFF"], title="Carga por área y urgencia")
pqrs.tipo_ia.value_counts().plot.pie(ax=ax[1], autopct="%1.0f%%", title="Tipo de PQRS (IA)", cmap="Purples"); ax[1].set_ylabel("")
pqrs.dias_habiles_restantes.clip(upper=15).plot.hist(bins=15, ax=ax[2], color="#9D4EDD", title="Días hábiles para vencer")
ax[2].axvline(3, color="red", ls="--"); plt.tight_layout(); plt.show()
print(f"🚨 Vencen en ≤3 días hábiles: {(pqrs.dias_habiles_restantes <= 3).sum()} · "
      f"Urgencia alta: {(pqrs.urgencia_ia == 'alta').sum()} · En casos masivos: {pqrs.masivo.sum()}")

---
## 5. ¿Cómo nos apoyamos en la IA? Impacto y riesgos

| Antes | Con el sistema |
|---|---|
| 300 h/mes leyendo y clasificando | El analista **revisa y corrige** una propuesta (~1 min por PQRS) |
| Casos graves enterrados en la cola | Bandeja ordenada por **urgencia y plazo legal** |
| 200 mensajes del mismo daño = 200 casos | **1 caso masivo**, una respuesta para todos |
| Respuestas desde cero | **Borrador** con datos del caso, revisado por un humano |
| "¿Por qué se clasificó así?" | **Explicación** palabra por palabra (oclusión) |

### Qué hace la IA y qué hace el humano
| La IA (Transformer) | El humano |
|---|---|
| Lee, entiende el contexto, propone tipo/área | Decide y corrige (cada corrección = dato para fine-tuning) |
| Detecta señales y agrupa casos | Define las reglas de negocio y los umbrales |
| Redacta un borrador | Responde de fondo y firma |

### ⚠️ Riesgos que hay que gestionar
- **Sesgos:** ¿clasifica peor a quien escribe con errores de ortografía o de manera informal? → evaluar por subgrupos.
- **Privacidad:** anonimizar antes de usar LLMs externos (Ley 1581 de 2012).
- **Alucinaciones:** el LLM nunca envía una respuesta sin revisión humana.
- **Deriva:** si cambian los productos o las tarifas, las etiquetas y el modelo se deben actualizar.

### 🏁 Reto
1. Agrega 5 PQRS nuevas "difíciles" (ironía, inglés, faltas de ortografía) y analiza qué falla.
2. Ajusta los pesos de `urgencia()` para lograr **100 % de recall en urgencia alta** sin marcar más del 40 % como alta.
3. Propón cómo usar las correcciones de los analistas para hacer **fine-tuning** de BETO cada mes.

### 🧑‍🏫 Preguntas frecuentes
- **"¿Por qué no usar ChatGPT para todo?"** → Costo por token × 3.000 PQRS, latencia, privacidad (datos personales fuera del país) y falta de control. Los modelos pequeños corren en la empresa y son medibles.
- **"¿La atención es la explicación del modelo?"** → No exactamente: la atención muestra **qué mira**, no **por qué decide**. Por eso usamos oclusión para explicar decisiones.
- **"¿Por qué 0.45 en el clustering?"** → Es un umbral de distancia coseno; se ajusta revisando grupos con el equipo. Más bajo = grupos más estrictos.
- **"¿Esto ya existe en empresas reales?"** → Sí: bancos, EPS, empresas de servicios públicos y entidades del Estado usan triage automático de PQRS y correos.